# 02 — Data Quality and Cleaning

This notebook audits data quality **after** integration and documents every cleaning decision.  
The raw source files (`data/raw/`) are **never modified**.

## Cleaning decisions
| Issue | Decision | Rationale |
|-------|----------|-----------|
| Team medals counted per athlete | Deduplicate on `[Games, NOC, Event, Medal]` before aggregation | Ensures medal counts reflect events won, not squad size |
| Historical NOCs (URS, GDR, etc.) | Map to modern successor ISO3 as *sensitivity* label | Needed for WDI join; kept as separate `mapping_source` column for transparency |
| WDI missingness (pre-1990 / small states) | Drop rows where `gdp_per_capita_usd` or `population` is NaN | Analytical sample; pre-clean file retained for missingness audit |
| Infinite `medals_per_million` | Not observed (no country has population = 0 in WDI) | Confirmed in audit |
| Duplicate rows | Checked after final join; zero exact duplicates found | Standard final sanity check |

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'tmp' / 'matplotlib'))

import pandas as pd
quality  = pd.read_csv(ROOT / 'data/processed/quality_audit.csv')
unmatched = pd.read_csv(ROOT / 'data/processed/unmatched_nocs.csv')
data = pd.read_csv(ROOT / 'data/processed/olympics_wealth_country_edition.csv')

print('Analytical rows:', len(data), '| exact duplicate rows:', data.duplicated().sum())
print('\n=== Feature-level missingness in analytical sample ===')
display(quality[quality.missing_count > 0])
print('\nUnmatched NOCs retained for audit:', len(unmatched))
display(unmatched)

### Missingness in the pre-cleaning full join (before WDI row-drop)

In [ ]:
preclean = pd.read_csv(ROOT / 'data/processed/olympics_wealth_preclean.csv')
miss = preclean.isna().mean().mul(100).sort_values(ascending=False)
miss = miss[miss > 0].rename('missing_pct').reset_index().rename(columns={'index': 'feature'})
print('Features with missing values before WDI restriction:')
display(miss)

### Duplicate-check: team events
Verify the deduplication rule worked correctly by spot-checking a known team event.

In [ ]:
athletes = pd.read_csv(ROOT / 'data/raw/athlete_events.csv')

# Example: 4x100m relay at 2000 Sydney — USA team has ~4 athletes, should become 1 medal row
relay = athletes[(athletes.Year == 2000) & (athletes.NOC == 'USA') & 
                 (athletes.Event == "Athletics Men's 4 x 100 metres Relay") & 
                 (athletes.Medal.notna())]
print(f"Athlete rows for USA 4x100 relay 2000: {len(relay)} athletes → 1 medal after dedup")
display(relay[['Name','NOC','Games','Event','Medal']])

# After deduplication
deduped = athletes.loc[(athletes.Year >= 1960) & athletes.Medal.notna()].drop_duplicates(['Games','NOC','Event','Medal'])
relay_deduped = deduped[(deduped.Year == 2000) & (deduped.NOC == 'USA') & 
                        (deduped.Event == "Athletics Men's 4 x 100 metres Relay")]
print(f"\nAfter dedup: {len(relay_deduped)} row(s)")
display(relay_deduped[['NOC','Games','Event','Medal']])

### Income quartile distribution

In [ ]:
print('Country-edition rows per income quartile:')
display(data.income_quartile.value_counts().sort_index().reset_index())

print('\nSample of Q4 highest-income entries:')
display(data[data.income_quartile == 'Q4 highest'][['Year','Season','NOC','medals','gdp_per_capita_usd','medals_per_million']].head(10))